In [1]:
import os

intermediates = [
    "district_table_2026-09-01.csv",
    "district_validation_2026-09-01.csv",
    "state_analysis_2026-09-01.csv",
    "localities_2026-09-01.csv",
    "death_notes_2026-09-01.csv",
    "cross_validation_2026-09-01.csv",
    "locality_count_validation_2026-09-01.csv",
]

for f in intermediates:
    path = os.path.join("data", "processed", f)
    if os.path.exists(path):
        os.remove(path)
        print(f"Deleted: {f}")

In [5]:
import os
import sqlite3

db_path = "data/idsp_kerala.db"
if os.path.exists(db_path):
    os.remove(db_path)

conn = sqlite3.connect(db_path)
cur = conn.cursor()
print("Fresh database created")

Fresh database created


In [6]:
cur.executescript("""
CREATE TABLE IF NOT EXISTS reports (
    report_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_date TEXT NOT NULL UNIQUE,
    period_type TEXT NOT NULL,
    source_url TEXT,
    filename TEXT NOT NULL,
    file_hash TEXT,
    ingested_at TEXT DEFAULT (datetime('now'))
);

CREATE TABLE IF NOT EXISTS observations (
    obs_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    geography_level TEXT NOT NULL,
    district_code TEXT NOT NULL,
    district_name TEXT NOT NULL,
    disease TEXT NOT NULL,
    metric TEXT NOT NULL,
    subtype TEXT,
    value REAL NOT NULL,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS locality_reports (
    loc_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    district_code TEXT NOT NULL,
    district_name TEXT NOT NULL,
    disease TEXT NOT NULL,
    district_reported_count REAL,
    locality_text TEXT NOT NULL,
    raw_text TEXT,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS death_notes (
    death_id INTEGER PRIMARY KEY AUTOINCREMENT,
    report_id INTEGER NOT NULL,
    district_code TEXT,
    district_name TEXT,
    disease TEXT,
    date_of_death TEXT,
    status TEXT NOT NULL,
    raw_text TEXT,
    source_page INTEGER,
    FOREIGN KEY (report_id) REFERENCES reports(report_id)
);

CREATE TABLE IF NOT EXISTS diseases (
    canonical_name TEXT PRIMARY KEY,
    aliases TEXT
);
""")
conn.commit()
print("Tables created")

Tables created


In [7]:
import hashlib
import pandas as pd

pdf_path = r"C:\Users\vinee\rag_chatbot_kerala\data\raw\daily\IDSP-Daily-Report-01.09.2026.pdf"
with open(pdf_path, "rb") as f:
    file_hash = hashlib.sha256(f.read()).hexdigest()

cur.execute("""
    INSERT OR IGNORE INTO reports (report_date, period_type, filename, file_hash)
    VALUES (?, ?, ?, ?)
""", ("2026-09-01", "daily", "IDSP-Daily-Report-01.09.2026.pdf", file_hash))
conn.commit()

cur.execute("SELECT report_id FROM reports WHERE report_date = '2026-09-01'")
report_id = cur.fetchone()[0]

# --- Observations ---
obs = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_district_observations_2026-09-01.csv")
for _, row in obs.iterrows():
    cur.execute("""
        INSERT INTO observations
        (report_id, geography_level, district_code, district_name,
         disease, metric, subtype, value, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row["geography_level"], row["district_code"],
          row["district_name"], row["disease"], row["metric"],
          row.get("subtype"), row["value"], row["source_page"]))

# --- Localities ---
loc = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_localities_2026-09-01.csv")
for _, row in loc.iterrows():
    cur.execute("""
        INSERT INTO locality_reports
        (report_id, district_code, district_name, disease,
         district_reported_count, locality_text, raw_text, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row["district_code"], row["district_name"],
          row["disease"], row.get("district_reported_count"),
          row["locality_text"], row.get("raw_text"), row["source_page"]))

# --- Death notes ---
deaths = pd.read_csv(r"C:\Users\vinee\rag_chatbot_kerala\data\processed\trusted_death_notes_2026-09-01.csv")
for _, row in deaths.iterrows():
    cur.execute("""
        INSERT INTO death_notes
        (report_id, district_code, district_name, disease,
         date_of_death, status, raw_text, source_page)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    """, (report_id, row.get("district_code"), row.get("district_name"),
          row.get("disease"), row.get("date_of_death"),
          row["status"], row.get("raw_text"), row["source_page"]))

conn.commit()
print(f"Loaded {len(obs)} observations, {len(loc)} localities, {len(deaths)} death notes")

KeyError: 'status'

In [8]:
for table in ["reports", "observations", "locality_reports", "death_notes"]:
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    print(f"{table}: {cur.fetchone()[0]} rows")

reports: 1 rows
observations: 406 rows
locality_reports: 24 rows
death_notes: 0 rows
